# Renda, inflação e desemprego: um painel de acompanhamento internacional
Cristiano Boni — EBAC — Projeto final / parceria Semantix

## Problema e finalidade
Organizações que acompanham condições econômicas precisam identificar onde renda e emprego evoluem de forma desfavorável, sem confundir aumento de preços com crescimento real. Como mudaram o PIB real por habitante, o desemprego e a inflação entre 2000 e 2024? Quais economias merecem investigação adicional?

O produto é um painel descritivo para triagem e monitoramento, não uma regra automática para alocar recursos. Não infere causalidade, risco individual ou recomendações financeiras. A unidade é economia–ano; os nomes e agrupamentos seguem a fonte. Agregados regionais do Banco Mundial são excluídos para evitar dupla contagem.

## Fontes e coleta
API pública do Banco Mundial, World Development Indicators. Indicadores: `NY.GDP.PCAP.KD` (PIB por habitante em US$ constantes de 2015), `SL.UEM.TOTL.ZS` (desemprego, estimativa modelada da OIT, % da força de trabalho) e `FP.CPI.TOTL.ZG` (inflação ao consumidor, % anual). Países e metadados vêm do endpoint `/country`.

Os JSON originais são preservados em `dados_brutos`. A coleta abaixo é opcional; `ATUALIZAR=False` reproduz o retrato anexado. Dados podem ser revisados pela fonte. Documentação: https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ATUALIZAR = False
urls = {'wb_countries':'https://api.worldbank.org/v2/country?format=json&per_page=500'}
for nome, indicador in [('wb_gdp','NY.GDP.PCAP.KD'),('wb_unemployment','SL.UEM.TOTL.ZS'),('wb_inflation','FP.CPI.TOTL.ZG')]:
    urls[nome] = f'https://api.worldbank.org/v2/country/all/indicator/{indicador}?format=json&date=2000:2024&per_page=20000'
if ATUALIZAR:
    for nome,url in urls.items():
        r = requests.get(url, timeout=90)
        r.raise_for_status()
        payload = r.json()
        assert payload[0]['pages'] == 1, 'Implementar paginação antes de usar um recorte maior.'
        Path(f'dados_brutos/{nome}.json').write_text(json.dumps(payload,ensure_ascii=False),encoding='utf-8')
raw = {n:json.loads(Path(f'dados_brutos/{n}.json').read_text(encoding='utf-8')) for n in urls}
print('URLs de coleta:')
print(json.dumps(urls,indent=2))
print('Atualização declarada pela fonte:',raw['wb_gdp'][0]['lastupdated'])


URLs de coleta:
{
  "wb_countries": "https://api.worldbank.org/v2/country?format=json&per_page=500",
  "wb_gdp": "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.KD?format=json&date=2000:2024&per_page=20000",
  "wb_unemployment": "https://api.worldbank.org/v2/country/all/indicator/SL.UEM.TOTL.ZS?format=json&date=2000:2024&per_page=20000",
  "wb_inflation": "https://api.worldbank.org/v2/country/all/indicator/FP.CPI.TOTL.ZG?format=json&date=2000:2024&per_page=20000"
}
Atualização declarada pela fonte: 2026-10-08


## Modelagem, limpeza e cruzamento
Cada indicador permanece em uma tabela longa com os campos naturais da API. A dimensão de economias preserva região, renda, capital e coordenadas. As chaves são ISO3 e ano; o cruzamento valida relações um para um. Ausências não viram zero e não são interpoladas. Valores extremos permanecem: inflação alta pode ser um evento real.

`data_referencia` é 1º de janeiro do ano de referência, uma convenção para o painel anual, e não a data de publicação da observação. Crescimento do PIB real per capita e variação do desemprego em pontos percentuais são derivados somente entre anos consecutivos com dados. As médias/medianas entre economias não têm ponderação populacional.


In [2]:
paises = pd.json_normalize(raw['wb_countries'][1])
paises = paises.loc[paises['region.value'].ne('Aggregates'), ['id','iso2Code','name','region.value','incomeLevel.value','capitalCity','longitude','latitude']].rename(columns={'id':'iso3','iso2Code':'iso2','name':'economia','region.value':'regiao','incomeLevel.value':'faixa_renda','capitalCity':'capital'})
for c in ['longitude','latitude']: paises[c] = pd.to_numeric(paises[c],errors='coerce')
assert paises['iso3'].is_unique
frames = {}
auditoria = []
for nome, campo in [('wb_gdp','pib_per_capita_real_usd'),('wb_unemployment','desemprego_pct'),('wb_inflation','inflacao_pct')]:
    base = pd.json_normalize(raw[nome][1]).rename(columns={'countryiso3code':'iso3','date':'ano','country.id':'iso2','country.value':'economia','indicator.id':'indicador','indicator.value':'descricao'})
    base = base[base['iso3'].isin(paises['iso3'])].copy()
    base['ano'] = pd.to_numeric(base['ano'],errors='raise').astype('int64')
    base['value'] = pd.to_numeric(base['value'],errors='coerce')
    base['data_referencia'] = pd.to_datetime(base['ano'].astype(str)+'-01-01')
    assert not base.duplicated(['iso3','ano']).any()
    auditoria.append({'tabela':nome,'linhas':len(base),'colunas':len(base.columns),'valores_ausentes':int(base['value'].isna().sum())})
    base.to_csv(nome+'.csv',index=False,encoding='utf-8-sig')
    frames[campo] = base[['iso3','ano','data_referencia','value']].rename(columns={'value':campo})
painel = frames['pib_per_capita_real_usd'].merge(frames['desemprego_pct'].drop(columns='data_referencia'),on=['iso3','ano'],validate='one_to_one').merge(frames['inflacao_pct'].drop(columns='data_referencia'),on=['iso3','ano'],validate='one_to_one').merge(paises,on='iso3',validate='many_to_one')
painel = painel.sort_values(['iso3','ano']).reset_index(drop=True)
grupo = painel.groupby('iso3')
consecutivo = grupo['ano'].diff().eq(1)
anterior_pib = grupo['pib_per_capita_real_usd'].shift()
painel['crescimento_pib_real_pct'] = ((painel['pib_per_capita_real_usd']/anterior_pib-1)*100).where(consecutivo & anterior_pib.gt(0))
painel['variacao_desemprego_pp'] = grupo['desemprego_pct'].diff().where(consecutivo)
assert not painel.duplicated(['iso3','ano']).any()
assert painel['desemprego_pct'].dropna().between(0,100).all()
assert painel['pib_per_capita_real_usd'].dropna().gt(0).all()
paises.to_csv('economias.csv',index=False,encoding='utf-8-sig')
painel.to_csv('painel_indicadores.csv',index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
painel.to_parquet('painel_indicadores.parquet',index=False)
display(pd.DataFrame(auditoria))
print('Painel:',painel.shape,'| Economias:',painel.iso3.nunique(),'| Período:',painel.ano.min(),painel.ano.max())
print('Ausências por variável:')
print(painel[['pib_per_capita_real_usd','desemprego_pct','inflacao_pct']].isna().sum().to_string())


Painel: (5425, 15) | Economias: 217 | Período: 2000 2024
Ausências por variável:
pib_per_capita_real_usd    266
desemprego_pct             759
inflacao_pct               897


,tabela,linhas,colunas,valores_ausentes
0,wb_gdp,5425,11,266
1,wb_unemployment,5425,11,759
2,wb_inflation,5425,11,897


## Cinco investigações exploratórias
1. Cobertura: quantas economias têm cada indicador por ano?
2. Tendência: como o Brasil evoluiu no período?
3. Distribuição e extremos: a inflação de 2024 apresenta caudas que distorcem a média?
4. Associação: PIB por habitante, inflação e desemprego estão correlacionados no recorte transversal de 2024?
5. Comparação regional: como varia a distribuição do desemprego em 2024?
6. Priorização: quais economias tiveram as maiores altas do desemprego entre 2023 e 2024?


In [3]:
sns.set_theme(style='whitegrid',font_scale=.9)
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.titleweight':'bold','axes.spines.top':False,'axes.spines.right':False})
Path('graficos').mkdir(exist_ok=True)
metricas=['pib_per_capita_real_usd','desemprego_pct','inflacao_pct']
cobertura=painel.groupby('ano')[metricas].count()
cobertura.to_csv('cobertura_por_ano.csv')
ax=cobertura.plot(title='Cobertura: economias com valores disponíveis',xlabel='Ano',ylabel='Número de economias',color=['#156082','#e87524','#3b8b67'])
ax.legend(['PIB por habitante','Desemprego','Inflação'],loc='lower left')
plt.tight_layout();plt.savefig('graficos/01_cobertura.png',dpi=140);plt.close()
br=painel[painel.iso3.eq('BRA')]
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,col,titulo in zip(axs,metricas,['PIB per capita real (US$ 2015)','Desemprego (%)','Inflação anual (%)']):
    ax.plot(br.ano,br[col],color='#156082');ax.set_title(titulo);ax.set_xlabel('Ano')
fig.suptitle('Brasil: trajetórias anuais de 2000 a 2024',weight='bold');fig.tight_layout();fig.savefig('graficos/02_brasil.png',dpi=140);plt.close(fig)
ano=painel[painel.ano.eq(2024)].copy()
infl=ano.inflacao_pct.dropna()
fig,ax=plt.subplots();ax.hist(infl,bins=28,color='#156082',edgecolor='white');ax.set(title='Inflação em 2024: distribuição completa',xlabel='Inflação anual (%)',ylabel='Economias');fig.tight_layout();fig.savefig('graficos/03_inflacao.png',dpi=140);plt.close(fig)
pares=ano[metricas].dropna()
corr=pares.corr(method='spearman');corr.to_csv('correlacoes_spearman_2024.csv')
fig,ax=plt.subplots();sns.heatmap(corr,annot=True,fmt='.2f',vmin=-1,vmax=1,cmap='RdBu_r',xticklabels=['PIB per capita','Desemprego','Inflação'],yticklabels=['PIB per capita','Desemprego','Inflação'],ax=ax);ax.set_title(f'Associações em 2024 — Spearman, n={len(pares)}');fig.tight_layout();fig.savefig('graficos/04_correlacao.png',dpi=140);plt.close(fig)
fig,ax=plt.subplots();sns.scatterplot(data=ano,x='pib_per_capita_real_usd',y='desemprego_pct',hue='faixa_renda',alpha=.7,ax=ax);ax.set_xscale('log');ax.set(title='PIB por habitante e desemprego em 2024',xlabel='PIB per capita real (US$ 2015, escala log)',ylabel='Desemprego (%)');ax.legend(title='Faixa de renda',fontsize=8);fig.tight_layout();fig.savefig('graficos/05_dispersao.png',dpi=140);plt.close(fig)
reg=ano.dropna(subset=['desemprego_pct']).copy();reg['regiao_curta']=reg.regiao.str.replace('Latin America & Caribbean ','América Latina/Caribe',regex=False).str.replace('Europe & Central Asia','Europa/Ásia Central',regex=False).str.replace('Middle East, North Africa, Afghanistan & Pakistan','Oriente Médio/Norte da África',regex=False).str.replace('Sub-Saharan Africa ','África Subsaariana',regex=False).str.replace('East Asia & Pacific','Leste Asiático/Pacífico',regex=False).str.replace('North America','América do Norte',regex=False).str.replace('South Asia','Sul da Ásia',regex=False)
fig,ax=plt.subplots();sns.boxplot(data=reg,y='regiao_curta',x='desemprego_pct',color='#7bb5c5',ax=ax);ax.set(title='Desemprego em 2024 por região',xlabel='Desemprego (%)',ylabel='');fig.tight_layout();fig.savefig('graficos/06_regioes.png',dpi=140);plt.close(fig)
altas=ano.nlargest(10,'variacao_desemprego_pp')[['economia','variacao_desemprego_pp','desemprego_pct']]
fig,ax=plt.subplots();ax.barh(altas.economia.iloc[::-1],altas.variacao_desemprego_pp.iloc[::-1],color='#e87524');ax.set(title='Maiores altas do desemprego: 2023 → 2024',xlabel='Variação em pontos percentuais');fig.tight_layout();fig.savefig('graficos/07_altas.png',dpi=140);plt.close(fig)
display(altas)
display(br.tail(5)[['ano']+metricas+['crescimento_pib_real_pct','variacao_desemprego_pp']])
stats={'economias':int(painel.iso3.nunique()),'linhas':len(painel),'tabelas':auditoria,'inflacao_2024_n':len(infl),'inflacao_2024_media':float(infl.mean()),'inflacao_2024_mediana':float(infl.median()),'inflacao_2024_max':float(infl.max()),'n_correlacao':len(pares),'corr_pib_desemprego':float(corr.loc['pib_per_capita_real_usd','desemprego_pct']),'maior_alta':altas.iloc[0].to_dict(),'brasil_2024':br.iloc[-1][['ano']+metricas+['crescimento_pib_real_pct','variacao_desemprego_pp']].to_dict(),'ausencias':painel[metricas].isna().sum().to_dict()}
Path('resultados.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(stats,ensure_ascii=False,indent=2))


{
  "economias": 217,
  "linhas": 5425,
  "tabelas": [
    {
      "tabela": "wb_gdp",
      "linhas": 5425,
      "colunas": 11,
      "valores_ausentes": 266
    },
    {
      "tabela": "wb_unemployment",
      "linhas": 5425,
      "colunas": 11,
      "valores_ausentes": 759
    },
    {
      "tabela": "wb_inflation",
      "linhas": 5425,
      "colunas": 11,
      "valores_ausentes": 897
    }
  ],
  "inflacao_2024_n": 176,
  "inflacao_2024_media": 7.407924533441152,
  "inflacao_2024_mediana": 3.014475769779985,
  "inflacao_2024_max": 219.883929014578,
  "n_correlacao": 164,
  "corr_pib_desemprego": 0.06183408750999945,
  "maior_alta": {
    "economia": "Panama",
    "variacao_desemprego_pp": 1.907000000000001,
    "desemprego_pct": 8.451
  },
  "brasil_2024": {
    "ano": 2024,
    "pib_per_capita_real_usd": 9566.74518687619,
    "desemprego_pct": 6.801,
    "inflacao_pct": 4.36746407652336,
    "crescimento_pib_real_pct": 3.000832918921925,
    "variacao_desemprego_pp": -1.14

,economia,variacao_desemprego_pp,desemprego_pct
3799,Panama,1.907,8.451
1574,Finland,1.251,8.400
1524,Estonia,1.222,7.600
2924,Luxembourg,1.216,6.400
174,Argentina,1.011,7.150
3724,New Zealand,0.990,4.724
824,Canada,0.936,6.351
4574,Sweden,0.789,8.400
2949,Latvia,0.435,6.900
774,Botswana,0.429,23.810


,ano,pib_per_capita_real_usd,desemprego_pct,inflacao_pct,crescimento_pib_real_pct,variacao_desemprego_pp
670,2020,8435.011433,13.697,3.211768,-3.835505,1.761
671,2021,8799.229458,13.158,8.301660,4.317932,-0.539
672,2022,9032.084816,9.231,9.280106,2.646315,-3.927
673,2023,9288.027015,7.947,4.593563,2.833700,-1.284
674,2024,9566.745187,6.801,4.367464,3.000833,-1.146


## Visualizações geradas
![Cobertura](graficos/01_cobertura.png)
![Brasil](graficos/02_brasil.png)
![Inflação](graficos/03_inflacao.png)
![Correlação](graficos/04_correlacao.png)
![Dispersão](graficos/05_dispersao.png)
![Regiões](graficos/06_regioes.png)
![Prioridades](graficos/07_altas.png)

## Limitações e ações
O painel apoia investigação, não prova o efeito de políticas. As observações são anuais, com cobertura e revisões diferentes por indicador. A comparação de países pequenos e grandes tem o mesmo peso e não representa a situação de uma pessoa típica no mundo. O desemprego é estimado e depende da definição da força de trabalho; inflação ao consumidor não mede diretamente o custo de vida de cada família.

Uma equipe de acompanhamento pode atualizar a coleta, conferir a cobertura antes de comparar anos, investigar aumentos persistentes de desemprego com fontes locais e usar a mediana e os quantis junto à média em séries com valores extremos. Não se deve definir prioridade policial, de crédito, de emprego ou de assistência a indivíduos com estes dados agregados.
